# E025 — Audio LPCC augmentation ablation

E020 adopted MFCC-optimal augmentation (noise SNR=20dB + speed ±10%) for
LPCC without verifying it is still optimal. This notebook ablates each
augmentation axis on top of the LPCC 39d baseline and compares to the
E020 result (3.33 ± 4.14%, min-DCF=0.0333).

Configs:
- `Baseline`     — original samples only
- `+Noise`       — +noise SNR=20dB
- `+Speed`       — +speed ±10%
- `+NoiseSpeed`  — E020 (+noise20 +speed)
- `+Pitch`       — +pitch shift ±{1,2} semitones
- `+HighSNR`     — +noise SNR=30dB (milder)
- `+AllMFCCOpt`  — +noise20 +speed (parity with +NoiseSpeed)

Everything else identical to E020: LPCC 13+Δ+ΔΔ+CMN, LPC order=12,
UBM 32, MAP r=16, LOSO seed=67, LLR scoring.

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    'target': '#E74C3C',
    'nontarget': '#2E86AB',
    'green': '#27AE60',
    'purple': '#8E44AD',
    'gray': '#95A5A6',
    'orange': '#E67E22',
}
CONFIG_ORDER = [
    'Baseline', '+Noise', '+Speed', '+NoiseSpeed',
    '+Pitch', '+HighSNR', '+AllMFCCOpt',
]
CONFIG_COLORS = {
    'Baseline':    '#95A5A6',
    '+Noise':      '#3498DB',
    '+Speed':      '#27AE60',
    '+NoiseSpeed': '#E74C3C',
    '+Pitch':      '#8E44AD',
    '+HighSNR':    '#E67E22',
    '+AllMFCCOpt': '#C0392B',
}
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
})

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
SEED = 67
print(f'{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target')

E020_REF = {'fold_eers': [9.17, 0.83, 0.00], 'mean': 3.33, 'std': 4.14, 'min_dcf_mean': 0.0333}

## 1. LPCC extraction + augmentation functions (copied from E020)

In [ ]:
def find_wav(stem, data_dir):
    for sf in ['target_train', 'target_dev', 'non_target_train', 'non_target_dev']:
        p = data_dir / sf / (stem + '.wav')
        if p.exists():
            return p
    raise FileNotFoundError(stem)


def extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    lpcc_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        lpcc_frames.append(cep)
    feat = np.array(lpcc_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat


def aug_noise_snr(y, rng, snr_db):
    p = np.mean(y**2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10**(snr_db/10)), len(y)).astype(y.dtype)


def aug_speed(y, rng):
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))


def aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


def make_variants(y, sr, rng, config):
    """Return list of (y_aug, sr). Always includes the original."""
    variants = [(y, sr)]
    if config == 'Baseline':
        pass
    elif config == '+Noise':
        variants.append((aug_noise_snr(y, rng, 20.0), sr))
    elif config == '+Speed':
        variants.append((aug_speed(y, rng), sr))
    elif config == '+NoiseSpeed':
        variants.append((aug_noise_snr(y, rng, 20.0), sr))
        variants.append((aug_speed(y, rng), sr))
    elif config == '+Pitch':
        variants.append((aug_pitch(y, sr, rng), sr))
    elif config == '+HighSNR':
        variants.append((aug_noise_snr(y, rng, 30.0), sr))
    elif config == '+AllMFCCOpt':
        variants.append((aug_noise_snr(y, rng, 20.0), sr))
        variants.append((aug_speed(y, rng), sr))
    else:
        raise ValueError(config)
    return variants


def extract_batch(df, data_dir, config, seed):
    rng = np.random.default_rng(seed)
    all_feat, all_labels = [], []
    for _, row in df.iterrows():
        y, sr = librosa.load(find_wav(row['stem'], data_dir), sr=None, mono=True)
        for y_aug, sr_aug in make_variants(y, sr, rng, config):
            feat = extract_lpcc(y_aug, sr_aug)
            all_feat.append(feat)
            all_labels.extend([row['label']] * len(feat))
    return np.vstack(all_feat), np.array(all_labels)


def score_utterance_lpcc(wav_path, adapted, ubm):
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    feat = extract_lpcc(y, sr)
    return float((adapted.score_samples(feat) - ubm.score_samples(feat)).mean())


print('Functions defined.')

## 2. UBM + MAP (same as E020)

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0


def train_ubm(X, n_components=32, seed=67):
    return GaussianMixture(
        n_components=n_components, covariance_type='diag',
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm, X_target, r=16.0):
    log_prob = ubm._estimate_log_prob(X_target)
    log_resp = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + r)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


print('UBM+MAP functions ready.')

## 3. LOSO CV loop over all 7 configs

For each config: LOSO 3-fold. Train UBM on non-target frames of the augmented
train fold; MAP-adapt on target frames of the train fold; score val
utterances on **original WAVs only** (no aug leakage).

In [ ]:
results = {}          # config -> {'fold_eers': [...], 'fold_dcfs': [...], 'oof_scores': np.array}

for config in CONFIG_ORDER:
    print(f"\n{'='*60}\nConfig: {config}\n{'='*60}")
    oof_scores = np.full(len(manifest), np.nan)
    fold_eers, fold_dcfs = [], []
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        X_train, y_train = extract_batch(train_df, DATA, config, seed=SEED + fold_id)
        X_nt = X_train[y_train == 0]
        X_t  = X_train[y_train == 1]
        ubm = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
        adapted = map_adapt(ubm, X_t, r=MAP_R)
        for idx, row in val_df.iterrows():
            oof_scores[idx] = score_utterance_lpcc(find_wav(row['stem'], DATA), adapted, ubm)
        vs = oof_scores[val_idx]
        vl = manifest.loc[val_idx, 'label'].to_numpy()
        eer, _ = compute_eer(vs[vl==1], vs[vl==0])
        mdcf, _ = compute_min_dcf(vs[vl==1], vs[vl==0])
        fold_eers.append(eer*100)
        fold_dcfs.append(mdcf)
        print(f'  fold {fold_id}: EER={eer*100:5.2f}%  min-DCF={mdcf:.4f}  (n_tframes={len(X_t)}, n_ntframes={len(X_nt)})')
    results[config] = {
        'fold_eers': fold_eers,
        'fold_dcfs': fold_dcfs,
        'oof_scores': oof_scores.copy(),
    }
    mean = np.mean(fold_eers); std = np.std(fold_eers)
    mdcf_mean = np.mean(fold_dcfs)
    print(f'  -> mean = {mean:.2f} ± {std:.2f} %  min-DCF = {mdcf_mean:.4f}')

print('\nAll configs complete.')

## 4. Results table

In [ ]:
rows = []
for cfg in CONFIG_ORDER:
    r = results[cfg]
    mean = float(np.mean(r['fold_eers'])); std = float(np.std(r['fold_eers']))
    rows.append({
        'Config': cfg,
        'F0 EER': r['fold_eers'][0],
        'F1 EER': r['fold_eers'][1],
        'F2 EER': r['fold_eers'][2],
        'Mean':   mean,
        'Std':    std,
        'min-DCF': float(np.mean(r['fold_dcfs'])),
    })
df_res = pd.DataFrame(rows)
best_idx = int(df_res['Mean'].idxmin())
winner = df_res.loc[best_idx, 'Config']

print(f"{'Config':<14} {'F0':>8} {'F1':>8} {'F2':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print('-' * 72)
# E020 reference row (identical in content to +NoiseSpeed but printed separately for trace)
print(f"{'E020 ref':<14} {E020_REF['fold_eers'][0]:>8.2f} {E020_REF['fold_eers'][1]:>8.2f} "
      f"{E020_REF['fold_eers'][2]:>8.2f} {E020_REF['mean']:>8.2f} {E020_REF['std']:>8.2f} "
      f"{E020_REF['min_dcf_mean']:>9.4f}")
print('-' * 72)
for i, row in df_res.iterrows():
    marker = ' *' if i == best_idx else '  '
    print(f"{row['Config']:<14} {row['F0 EER']:>8.2f} {row['F1 EER']:>8.2f} {row['F2 EER']:>8.2f} "
          f"{row['Mean']:>8.2f} {row['Std']:>8.2f} {row['min-DCF']:>9.4f}{marker}")
print('-' * 72)
print(f"\nWinner: {winner}  (EER={df_res.loc[best_idx,'Mean']:.2f} ± {df_res.loc[best_idx,'Std']:.2f}%, "
      f"min-DCF={df_res.loc[best_idx,'min-DCF']:.4f})")
print(f"Delta vs E020 +NoiseSpeed (3.33 ± 4.14%): {df_res.loc[best_idx,'Mean'] - 3.33:+.2f}pp")

df_res

## 5. Horizontal bar chart with E020 reference line

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
y_pos = np.arange(len(CONFIG_ORDER))
means = [float(np.mean(results[c]['fold_eers'])) for c in CONFIG_ORDER]
stds  = [float(np.std(results[c]['fold_eers']))  for c in CONFIG_ORDER]
colors = [CONFIG_COLORS[c] for c in CONFIG_ORDER]

best = int(np.argmin(means))
bars = ax.barh(y_pos, means, xerr=stds, color=colors, alpha=0.85,
               error_kw=dict(elinewidth=1.8, ecolor='#444'), capsize=4)
bars[best].set_edgecolor('gold')
bars[best].set_linewidth(3)

for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(m + s + 0.15, i, f'{m:.2f} ± {s:.2f}', va='center', fontsize=9)

# E020 reference line
ax.axvline(E020_REF['mean'], color='black', ls='--', lw=1.2, alpha=0.6,
           label=f"E020 ref = {E020_REF['mean']:.2f}%")

ax.set_yticks(y_pos)
ax.set_yticklabels(CONFIG_ORDER)
ax.invert_yaxis()
ax.set_xlabel('EER mean ± std [%]')
ax.set_title('E025 — LPCC augmentation ablation (LOSO, seed=67)')
ax.legend(loc='lower right', fontsize=9)
plt.tight_layout()
plt.show()

## 6. DET curves (OOF across all folds per config)

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos = [scipy_norm.ppf(t) for t in ticks]
tick_lab = [f'{int(t*100)}' for t in ticks]

fig, ax = plt.subplots(figsize=(8, 7))
for cfg in CONFIG_ORDER:
    s = results[cfg]['oof_scores']
    m = ~np.isnan(s)
    eer_oof, _ = compute_eer(s[m & (y_all==1)], s[m & (y_all==0)])
    fpr, tpr, _ = roc_curve(y_all[m], s[m])
    far_c = np.clip(fpr, 1e-4, 1-1e-4)
    frr_c = np.clip(1-tpr, 1e-4, 1-1e-4)
    lw = 2.8 if cfg == CONFIG_ORDER[int(np.argmin([np.mean(results[c]['fold_eers']) for c in CONFIG_ORDER]))] else 1.6
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=CONFIG_COLORS[cfg], lw=lw,
            label=f'{cfg}  EER_OOF={eer_oof*100:.2f}%')

ax.plot(tick_pos, tick_pos, 'k--', lw=1, alpha=0.5)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_lab)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_lab)
ax.set_xlabel('FAR [%]')
ax.set_ylabel('FRR [%]')
ax.set_title('E025 — DET curves per augmentation config (LPCC OOF)')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()

## 7. Summary for the experiment log

In [ ]:
print('markdown_table:')
print('| Config       | F0 EER | F1 EER | F2 EER | Mean ± std      | min-DCF |')
print('| ------------ | ------ | ------ | ------ | --------------- | ------- |')
print(f'| E020 ref     | {E020_REF["fold_eers"][0]:>6.2f} | {E020_REF["fold_eers"][1]:>6.2f} '
      f'| {E020_REF["fold_eers"][2]:>6.2f} | {E020_REF["mean"]:>4.2f} ± {E020_REF["std"]:.2f}    '
      f'| {E020_REF["min_dcf_mean"]:.4f} |')
for i, row in df_res.iterrows():
    star = ' **' if i == best_idx else '    '
    name = f'**{row["Config"]}**' if i == best_idx else row['Config']
    print(f'|{star}{name:<10}{star}| {row["F0 EER"]:>6.2f} | {row["F1 EER"]:>6.2f} | '
          f'{row["F2 EER"]:>6.2f} | {row["Mean"]:>4.2f} ± {row["Std"]:.2f}    | {row["min-DCF"]:.4f} |')
print()
print(f'Winner: {winner}  EER = {df_res.loc[best_idx,"Mean"]:.2f} ± {df_res.loc[best_idx,"Std"]:.2f} %  '
      f'min-DCF = {df_res.loc[best_idx,"min-DCF"]:.4f}')